# 05 · Meal Plan — Rule-Based Assignment
**CalixAI** — The meal plan is assigned via a deterministic rule function (no ML model).
Goal + fitness level uniquely determine the plan key, so training a classifier would achieve
100% accuracy trivially (no generalisation). The rule function is embedded directly in `app.py`.

In [ ]:
import pandas as pd
import os
import pickle

BASE = os.path.abspath('..')
df   = pd.read_csv(os.path.join(BASE,'data','raw','calisthenics_dataset.csv'))
print('Dataset shape:', df.shape)
print('Meal plan classes:', sorted(df['meal_plan'].unique()))

## Why no ML model?

Meal plan assignment is 100% deterministic: every `(goal, fitness_level)` pair maps to exactly one
plan key with no overlap or ambiguity. A classifier trained on this signal will memorise the lookup
table and return 100% accuracy — but it learns nothing generalisable.

Replacing the ML model with a direct rule function gives:
- Zero inference latency (dict lookup vs model prediction)
- No pkl files to manage or version
- Identical accuracy (100%) with full explainability

In [ ]:
# The exact rule function used in app.py
MEAL_PLAN_RULES = {
    ('Fat Loss',    'Beginner'):     'fl_beginner',
    ('Fat Loss',    'Intermediate'): 'fl_intermediate',
    ('Fat Loss',    'Advanced'):     'fl_advanced',
    ('Muscle Gain', 'Beginner'):     'mg_beginner',
    ('Muscle Gain', 'Intermediate'): 'mg_intermediate',
    ('Muscle Gain', 'Advanced'):     'mg_advanced',
    ('Maintenance', 'Beginner'):     'mt_standard',
    ('Maintenance', 'Intermediate'): 'mt_standard',
    ('Maintenance', 'Advanced'):     'mt_standard',
}

# Verify against dataset — should be 100%
df['predicted_plan'] = df.apply(
    lambda r: MEAL_PLAN_RULES.get((r['goal'], r['fitness_level']), 'mt_standard'), axis=1
)
match = (df['predicted_plan'] == df['meal_plan']).mean()
print(f'Rule accuracy on full dataset: {match*100:.2f}%')
print()
print('Plan distribution:')
print(df['meal_plan'].value_counts().to_string())

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style='darkgrid')

counts = df['meal_plan'].value_counts()
counts.plot(kind='bar', color=sns.color_palette('muted', len(counts)))
plt.title('Meal Plan Distribution (rule-assigned)')
plt.ylabel('Count')
plt.xticks(rotation=30)
plt.tight_layout()
plt.show()
print('No pkl files saved — rule function is embedded in app.py.')